In [2]:
import kinematics as kn
import model as mod
import pyplot as plot
import bvhio
import numpy as np


from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation as R
from kinematics import Joint

In [3]:
# Normalization

calculated_spine_len = None

def facing_correction(root: Joint) -> np.ndarray:
    y_angle = R.from_quat(root.quat).as_euler('yxz')[0]
    return R.from_euler('y', -y_angle).as_quat()

def get_spine_length(root: Joint, spine_chain: list[str]) -> float:
    """Calculates the total length of the spine by summing bone lengths."""

    global calculated_spine_len

    if calculated_spine_len is not None:
        return calculated_spine_len

    total_length = 0.0
    
    for i in range(1, len(spine_chain)):
        parent = root[spine_chain[i-1]]
        child  = root[spine_chain[i]]
        
        if parent is None or child is None:
            print(f"Warning: Joint missing in spine chain ({spine_chain[i-1]} or {spine_chain[i]})")
            continue
            
        # The bone length is the physical distance between the parent and child
        # Since child.offset is the local translation from the parent, 
        # its magnitude is exactly the bone length!
        bone_length = np.linalg.norm(child.offset)
        total_length += bone_length
    
    calculated_spine_len = total_length

    return total_length

def normalize_skeleton_scale(root: Joint, target_spine_length: float = 1.0) -> Joint:
    """
    Scales the entire skeleton proportionally so its spine matches the target length.
    """
    # 1. Define the spine hierarchy exactly as it appears in the BVH file
    SPINE_CHAIN = [
        "root", 
        "torso_1", 
        "torso_2", 
        "torso_3", 
        "torso_4", 
        "torso_5", 
        "torso_6", 
        "torso_7", 
        "neck_1"
    ]
    
    # 2. Get current spine length
    current_length = get_spine_length(root, SPINE_CHAIN)
    if current_length < 1e-6:
        print("Error: Spine length is near zero. Check your joint IDs.")
        return root
        
    # 3. Calculate the universal scalar
    scale_factor = target_spine_length / current_length

    # 4. Apply the scalar to EVERY joint's local offset
    def _scale_walk(joint: Joint):
        joint.offset = joint.offset * scale_factor
        for child in joint.children:
            _scale_walk(child)
            
    _scale_walk(root)

    return root

def normalize(root: Joint) -> Joint:
    root.offset = np.array([0, 0, 0])
    root.quat   = root.transform.rotate(facing_correction(root))
    return normalize_skeleton_scale(root)

In [8]:
# Graph plotter

print("Loading model and normalizing...")
frames = mod.load_model("../assets/MCPM_20260423_140006.bvh", frameMap=normalize)

print("Plotting...")
ani = plot.plot_skeleton(frames)

Loading model and normalizing...
Plotting...


In [ ]:
# Profiling / Benchmarking
# import cProfile
# import pstats

# with cProfile.Profile() as pr:
#     frames = mod.load_model("../assets/MCPM_20260423_140006.bvh", frameMap=normalize)

# stats = pstats.Stats(pr)
# stats.sort_stats('cumulative')
# stats.print_stats(15)  # top 15 slowest calls